# 02 Data Quality

Profiles the three raw tables from `01_data_setup.ipynb` and reports whether they are fit for analysis.

| Section | Dimension |
| --- | --- |
| 3 | Completeness |
| 4 | Uniqueness |
| 5 | Identifier validity |
| 6 | Timestamp validity |
| 7 | Delay accuracy and outliers |
| 8 | Referential integrity |
| 9 | Calendar coverage |
| 10 | Scorecard, issues and recommendations |

Read-only: nothing is modified or persisted. Issues are detected here and corrected downstream. Run notebook 01 first.

## 1. Setup

Consumes the database built by notebook 01. Fails fast if the file or a table is missing.

In [1]:
from pathlib import Path

import duckdb

# Resolve the project root from data/ so the notebook runs from notebooks/ or from the repo root.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
DB_PATH = PROJECT_ROOT / "railpulse.duckdb"

TABLES = ["train_run_events", "train_master", "station_master"]

# Fail fast with an actionable message. Rebuilding the database is notebook 01's job, not this one's.
if not DB_PATH.is_file():
    raise FileNotFoundError(f"{DB_PATH} not found. Run notebooks/01_data_setup.ipynb first.")

# Read-only connection: an audit must not be able to mutate the data it audits.
con = duckdb.connect(str(DB_PATH), read_only=True)

# Guard against a partial load from notebook 01.
existing = {row[0] for row in con.sql("SHOW TABLES").fetchall()}
missing = [table for table in TABLES if table not in existing]
assert not missing, f"Missing tables {missing}. Run notebooks/01_data_setup.ipynb first."

print(f"Connected to {DB_PATH.relative_to(PROJECT_ROOT)} (read-only)")
print(f"DuckDB version: {duckdb.__version__}")

Connected to railpulse.duckdb (read-only)
DuckDB version: 1.5.6


In [2]:
def show(sql, max_rows=50):
    """Execute a query and render the result wide enough that columns are not truncated."""
    con.sql(sql).show(max_width=220, max_rows=max_rows)


# Baseline row counts: the denominators for the checks that follow.
show("""
    SELECT 'train_run_events' AS table_name, COUNT(*) AS row_count FROM train_run_events
    UNION ALL
    SELECT 'train_master', COUNT(*) FROM train_master
    UNION ALL
    SELECT 'station_master', COUNT(*) FROM station_master
""")

┌──────────────────┬───────────┐
│    table_name    │ row_count │
│     varchar      │   int64   │
├──────────────────┼───────────┤
│ train_run_events │      2460 │
│ train_master     │        10 │
│ station_master   │        20 │
└──────────────────┴───────────┘



## 2. Helper view

Derived columns shared by the checks, defined once: origin and terminus flags, recomputed delays, dwell minutes. Same grain as `train_run_events`. Temporary.

In [3]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW events_checked AS
    -- Grain is unchanged from train_run_events: one row per run per stop, no rows added or dropped.
    SELECT
        *,
        -- Derived from the sequence bounds, not hard-coded to 1 and n, so the flags survive a sequence gap.
        station_sequence = MIN(station_sequence) OVER run AS is_origin,
        station_sequence = MAX(station_sequence) OVER run AS is_terminus,
        -- Recomputed from the timestamps so the stored delay columns can be audited against them (7.1).
        date_diff('minute', scheduled_arrival, actual_arrival) AS calc_arrival_delay_min,
        date_diff('minute', scheduled_departure, actual_departure) AS calc_departure_delay_min,
        date_diff('minute', scheduled_arrival, scheduled_departure) AS scheduled_dwell_min,
        date_diff('minute', actual_arrival, actual_departure) AS actual_dwell_min
    FROM train_run_events
    WINDOW run AS (PARTITION BY service_id)
""")
print("Created temporary view: events_checked")

Created temporary view: events_checked


## 3. Missing values

Per column: `NULL` values, blank strings, and padded strings that would break joins. Pass: all counts are 0.

### 3.1 `train_run_events`

In [4]:
show("""
    SELECT
        column_name,
        COUNT(*) AS row_count,
        COUNT(*) - COUNT(value) AS null_count,
        COUNT(*) FILTER (WHERE trim(value) = '') AS blank_count,
        COUNT(*) FILTER (WHERE value <> trim(value)) AS padded_count
    -- Cast to VARCHAR so mixed types share one UNPIVOT. INCLUDE NULLS is required: UNPIVOT drops NULLs by default.
    FROM (SELECT COLUMNS(*)::VARCHAR FROM train_run_events)
    UNPIVOT INCLUDE NULLS (value FOR column_name IN (COLUMNS(*)))
    GROUP BY column_name
    ORDER BY null_count DESC, column_name
""")

┌───────────────────────────┬───────────┬────────────┬─────────────┬──────────────┐
│        column_name        │ row_count │ null_count │ blank_count │ padded_count │
│          varchar          │   int64   │   int64    │    int64    │    int64     │
├───────────────────────────┼───────────┼────────────┼─────────────┼──────────────┤
│ actual_arrival            │      2460 │          0 │           0 │            0 │
│ actual_departure          │      2460 │          0 │           0 │            0 │
│ arrival_delay_min         │      2460 │          0 │           0 │            0 │
│ data_status               │      2460 │          0 │           0 │            0 │
│ departure_delay_min       │      2460 │          0 │           0 │            0 │
│ scheduled_arrival         │      2460 │          0 │           0 │            0 │
│ scheduled_departure       │      2460 │          0 │           0 │            0 │
│ service_date              │      2460 │          0 │           0 │        

### 3.2 `train_master`

In [5]:
show("""
    SELECT
        column_name,
        COUNT(*) AS row_count,
        COUNT(*) - COUNT(value) AS null_count,
        COUNT(*) FILTER (WHERE trim(value) = '') AS blank_count,
        COUNT(*) FILTER (WHERE value <> trim(value)) AS padded_count
    -- Cast to VARCHAR so mixed types share one UNPIVOT. INCLUDE NULLS is required: UNPIVOT drops NULLs by default.
    FROM (SELECT COLUMNS(*)::VARCHAR FROM train_master)
    UNPIVOT INCLUDE NULLS (value FOR column_name IN (COLUMNS(*)))
    GROUP BY column_name
    ORDER BY null_count DESC, column_name
""")

┌─────────────────────┬───────────┬────────────┬─────────────┬──────────────┐
│     column_name     │ row_count │ null_count │ blank_count │ padded_count │
│       varchar       │   int64   │   int64    │    int64    │    int64     │
├─────────────────────┼───────────┼────────────┼─────────────┼──────────────┤
│ destination_station │        10 │          0 │           0 │            0 │
│ origin_station      │        10 │          0 │           0 │            0 │
│ route_note          │        10 │          0 │           0 │            0 │
│ service_type        │        10 │          0 │           0 │            0 │
│ train_id            │        10 │          0 │           0 │            0 │
│ train_name          │        10 │          0 │           0 │            0 │
└─────────────────────┴───────────┴────────────┴─────────────┴──────────────┘



### 3.3 `station_master`

In [6]:
show("""
    SELECT
        column_name,
        COUNT(*) AS row_count,
        COUNT(*) - COUNT(value) AS null_count,
        COUNT(*) FILTER (WHERE trim(value) = '') AS blank_count,
        COUNT(*) FILTER (WHERE value <> trim(value)) AS padded_count
    -- Cast to VARCHAR so mixed types share one UNPIVOT. INCLUDE NULLS is required: UNPIVOT drops NULLs by default.
    FROM (SELECT COLUMNS(*)::VARCHAR FROM station_master)
    UNPIVOT INCLUDE NULLS (value FOR column_name IN (COLUMNS(*)))
    GROUP BY column_name
    ORDER BY null_count DESC, column_name
""")

┌──────────────┬───────────┬────────────┬─────────────┬──────────────┐
│ column_name  │ row_count │ null_count │ blank_count │ padded_count │
│   varchar    │   int64   │   int64    │    int64    │    int64     │
├──────────────┼───────────┼────────────┼─────────────┼──────────────┤
│ region       │        20 │          0 │           0 │            0 │
│ station_id   │        20 │          0 │           0 │            0 │
│ station_name │        20 │          0 │           0 │            0 │
└──────────────┴───────────┴────────────┴─────────────┴──────────────┘



### 3.4 Placeholder values

`none` is a valid disruption type only when the disruption flag is `false`.

In [7]:
show("""
    -- Flag and type state the same fact twice, so any disagreement means one of them is wrong.
    SELECT
        simulated_disruption_flag,
        simulated_disruption_type,
        COUNT(*) AS row_count,
        CASE
            WHEN simulated_disruption_flag AND simulated_disruption_type = 'none' THEN 'ISSUE: flag without type'
            WHEN NOT simulated_disruption_flag AND simulated_disruption_type <> 'none' THEN 'ISSUE: type without flag'
            ELSE 'consistent'
        END AS assessment
    FROM train_run_events
    GROUP BY ALL
    ORDER BY simulated_disruption_flag, simulated_disruption_type
""")

┌───────────────────────────┬───────────────────────────┬───────────┬────────────┐
│ simulated_disruption_flag │ simulated_disruption_type │ row_count │ assessment │
│          boolean          │          varchar          │   int64   │  varchar   │
├───────────────────────────┼───────────────────────────┼───────────┼────────────┤
│ false                     │ none                      │      2324 │ consistent │
│ true                      │ operational               │        28 │ consistent │
│ true                      │ signal                    │        43 │ consistent │
│ true                      │ track                     │        30 │ consistent │
│ true                      │ weather                   │        35 │ consistent │
└───────────────────────────┴───────────────────────────┴───────────┴────────────┘



## 4. Duplicate records

### 4.1 Duplicate rows and duplicate keys

Identical rows indicate a double load. Repeated keys fan out joins. Pass: both are 0.

In [8]:
show("""
    -- duplicate_rows > 0 suggests a double load. duplicate_keys > 0 alone suggests conflicting versions of a record.
    SELECT
        'train_run_events' AS table_name,
        'service_id, station_sequence' AS key_columns,
        COUNT(*) AS row_count,
        COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM train_run_events)) AS duplicate_rows,
        COUNT(*) - COUNT(DISTINCT (service_id, station_sequence)) AS duplicate_keys
    FROM train_run_events
    UNION ALL
    SELECT 'train_master', 'train_id', COUNT(*),
        COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM train_master)),
        COUNT(*) - COUNT(DISTINCT train_id)
    FROM train_master
    UNION ALL
    SELECT 'station_master', 'station_id', COUNT(*),
        COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM station_master)),
        COUNT(*) - COUNT(DISTINCT station_id)
    FROM station_master
""")

┌──────────────────┬──────────────────────────────┬───────────┬────────────────┬────────────────┐
│    table_name    │         key_columns          │ row_count │ duplicate_rows │ duplicate_keys │
│     varchar      │           varchar            │   int64   │     int64      │     int64      │
├──────────────────┼──────────────────────────────┼───────────┼────────────────┼────────────────┤
│ train_run_events │ service_id, station_sequence │      2460 │              0 │              0 │
│ train_master     │ train_id                     │        10 │              0 │              0 │
│ station_master   │ station_id                   │        20 │              0 │              0 │
└──────────────────┴──────────────────────────────┴───────────┴────────────────┴────────────────┘



### 4.2 Duplicates that the key does not catch

Business-level duplicates: a station repeated in a run, two runs of a train on one date, one name under two IDs.

In [9]:
show("""
    -- Business-level duplicates: unique on the technical key, yet the same real-world event recorded twice.
    SELECT 'Station repeated within a run' AS check_name, COUNT(*) AS groups_found
    FROM (
        SELECT service_id, station_id
        FROM train_run_events
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    UNION ALL
    SELECT 'Train with more than one run on a date', COUNT(*)
    FROM (
        SELECT train_id, service_date
        FROM train_run_events
        GROUP BY ALL
        HAVING COUNT(DISTINCT service_id) > 1
    )
    UNION ALL
    SELECT 'Train name used by more than one train_id', COUNT(*)
    FROM (SELECT train_name FROM train_master GROUP BY ALL HAVING COUNT(*) > 1)
    UNION ALL
    SELECT 'Station name used by more than one station_id', COUNT(*)
    FROM (SELECT station_name FROM station_master GROUP BY ALL HAVING COUNT(*) > 1)
""")

┌───────────────────────────────────────────────┬──────────────┐
│                  check_name                   │ groups_found │
│                    varchar                    │    int64     │
├───────────────────────────────────────────────┼──────────────┤
│ Station repeated within a run                 │            0 │
│ Train with more than one run on a date        │            0 │
│ Train name used by more than one train_id     │            0 │
│ Station name used by more than one station_id │            0 │
└───────────────────────────────────────────────┴──────────────┘



## 5. Train, station and run identifiers

### 5.1 ID format

Expected: `TR` + 3 digits, `ST` + 3 digits, `RUN` + 6 digits. A variant spelling splits one entity across two keys.

In [10]:
show("""
    -- regexp_full_match anchors the whole value, so stray whitespace or a case change fails the check.
    SELECT 'train_run_events' AS table_name, 'service_id' AS column_name, 'RUN000000' AS expected_format,
        COUNT(DISTINCT service_id) AS distinct_ids,
        COUNT(*) FILTER (WHERE NOT regexp_full_match(service_id, 'RUN[0-9]{6}')) AS invalid_rows
    FROM train_run_events
    UNION ALL
    SELECT 'train_run_events', 'train_id', 'TR000', COUNT(DISTINCT train_id),
        COUNT(*) FILTER (WHERE NOT regexp_full_match(train_id, 'TR[0-9]{3}'))
    FROM train_run_events
    UNION ALL
    SELECT 'train_run_events', 'station_id', 'ST000', COUNT(DISTINCT station_id),
        COUNT(*) FILTER (WHERE NOT regexp_full_match(station_id, 'ST[0-9]{3}'))
    FROM train_run_events
    UNION ALL
    SELECT 'train_master', 'train_id', 'TR000', COUNT(DISTINCT train_id),
        COUNT(*) FILTER (WHERE NOT regexp_full_match(train_id, 'TR[0-9]{3}'))
    FROM train_master
    UNION ALL
    SELECT 'station_master', 'station_id', 'ST000', COUNT(DISTINCT station_id),
        COUNT(*) FILTER (WHERE NOT regexp_full_match(station_id, 'ST[0-9]{3}'))
    FROM station_master
""")

┌──────────────────┬─────────────┬─────────────────┬──────────────┬──────────────┐
│    table_name    │ column_name │ expected_format │ distinct_ids │ invalid_rows │
│     varchar      │   varchar   │     varchar     │    int64     │    int64     │
├──────────────────┼─────────────┼─────────────────┼──────────────┼──────────────┤
│ train_run_events │ service_id  │ RUN000000       │          426 │            0 │
│ train_run_events │ train_id    │ TR000           │           10 │            0 │
│ train_run_events │ station_id  │ ST000           │           19 │            0 │
│ train_master     │ train_id    │ TR000           │           10 │            0 │
│ station_master   │ station_id  │ ST000           │           20 │            0 │
└──────────────────┴─────────────┴─────────────────┴──────────────┴──────────────┘



### 5.2 Consistency within a run

One train, one service date and a gap-free stop sequence per `service_id`.

In [11]:
show("""
    -- Collapse to run grain first: every rule below is a property of a run, not of a stop.
    WITH runs AS (
        SELECT
            service_id,
            COUNT(*) AS stops,
            COUNT(DISTINCT train_id) AS train_ids,
            COUNT(DISTINCT service_date) AS service_dates,
            MIN(station_sequence) AS first_sequence,
            MAX(station_sequence) AS last_sequence
        FROM train_run_events
        GROUP BY service_id
    )
    SELECT
        COUNT(*) AS runs,
        COUNT(*) FILTER (WHERE train_ids > 1) AS runs_with_mixed_train_ids,
        COUNT(*) FILTER (WHERE service_dates > 1) AS runs_with_mixed_service_dates,
        -- last_sequence = stops holds only for a gap-free 1..n sequence.
        COUNT(*) FILTER (WHERE first_sequence <> 1 OR last_sequence <> stops) AS runs_with_sequence_gaps,
        COUNT(*) FILTER (WHERE stops < 2) AS runs_with_fewer_than_2_stops
    FROM runs
""")

┌───────┬───────────────────────────┬───────────────────────────────┬─────────────────────────┬──────────────────────────────┐
│ runs  │ runs_with_mixed_train_ids │ runs_with_mixed_service_dates │ runs_with_sequence_gaps │ runs_with_fewer_than_2_stops │
│ int64 │           int64           │             int64             │          int64          │            int64             │
├───────┼───────────────────────────┼───────────────────────────────┼─────────────────────────┼──────────────────────────────┤
│   426 │                         0 │                             0 │                       0 │                            0 │
└───────┴───────────────────────────┴───────────────────────────────┴─────────────────────────┴──────────────────────────────┘



### 5.3 Route consistency per train

Pass: one route variant per train.

In [12]:
show("""
    WITH run_routes AS (
        SELECT
            service_id,
            train_id,
            -- Route signature: ORDER BY inside string_agg makes it deterministic.
            string_agg(station_id, ' > ' ORDER BY station_sequence) AS route
        FROM train_run_events
        GROUP BY service_id, train_id
    )
    SELECT
        train_id,
        COUNT(*) AS runs,
        COUNT(DISTINCT route) AS route_variants,
        MIN(route) AS route
    FROM run_routes
    GROUP BY train_id
    ORDER BY train_id
""")

┌──────────┬───────┬────────────────┬───────────────────────────────────────────────────────┐
│ train_id │ runs  │ route_variants │                         route                         │
│ varchar  │ int64 │     int64      │                        varchar                        │
├──────────┼───────┼────────────────┼───────────────────────────────────────────────────────┤
│ TR001    │    39 │              1 │ ST001 > ST002 > ST013 > ST012 > ST011                 │
│ TR002    │    45 │              1 │ ST001 > ST015 > ST014 > ST013 > ST012 > ST011         │
│ TR003    │    45 │              1 │ ST001 > ST002 > ST003 > ST016 > ST017                 │
│ TR004    │    45 │              1 │ ST001 > ST003 > ST004 > ST005 > ST017                 │
│ TR005    │    39 │              1 │ ST001 > ST002 > ST015 > ST014 > ST019 > ST020 > ST018 │
│ TR006    │    39 │              1 │ ST001 > ST002 > ST003 > ST004 > ST007 > ST008 > ST010 │
│ TR007    │    39 │              1 │ ST001 > ST003 > ST004 

## 6. Arrival and departure timestamps

### 6.1 Data types and range

All timestamp columns must be typed `TIMESTAMP` and fall in the expected period.

In [13]:
show("""
    -- A timestamp column that failed type inference at load would surface here as VARCHAR.
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'train_run_events'
      AND (column_name LIKE '%arrival' OR column_name LIKE '%departure' OR column_name = 'service_date')
    ORDER BY ordinal_position
""")

┌─────────────────────┬───────────┐
│     column_name     │ data_type │
│       varchar       │  varchar  │
├─────────────────────┼───────────┤
│ service_date        │ DATE      │
│ scheduled_arrival   │ TIMESTAMP │
│ actual_arrival      │ TIMESTAMP │
│ scheduled_departure │ TIMESTAMP │
│ actual_departure    │ TIMESTAMP │
└─────────────────────┴───────────┘



In [14]:
show("""
    SELECT
        MIN(service_date) AS first_service_date,
        MAX(service_date) AS last_service_date,
        LEAST(MIN(scheduled_arrival), MIN(actual_arrival)) AS earliest_timestamp,
        GREATEST(MAX(scheduled_departure), MAX(actual_departure)) AS latest_timestamp
    FROM train_run_events
""")

┌────────────────────┬───────────────────┬─────────────────────┬─────────────────────┐
│ first_service_date │ last_service_date │ earliest_timestamp  │  latest_timestamp   │
│        date        │       date        │      timestamp      │      timestamp      │
├────────────────────┼───────────────────┼─────────────────────┼─────────────────────┤
│ 2026-08-01         │ 2026-09-14        │ 2026-08-01 06:35:00 │ 2026-09-15 05:33:00 │
└────────────────────┴───────────────────┴─────────────────────┴─────────────────────┘



### 6.2 Timestamps against the service date

A run may cross midnight, so the day after the service date is valid. Anything earlier or later is not.

In [15]:
show("""
    -- Scheduled time is the reference: it is independent of delay, so the check isolates calendar errors.
    SELECT
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE scheduled_arrival::DATE = service_date + 1) AS rows_after_midnight,
        COUNT(*) FILTER (WHERE scheduled_arrival::DATE < service_date) AS rows_before_service_date,
        COUNT(*) FILTER (WHERE scheduled_arrival::DATE > service_date + 1) AS rows_more_than_1_day_later,
        COUNT(*) FILTER (WHERE is_origin AND scheduled_departure::DATE <> service_date) AS origins_not_on_service_date
    FROM events_checked
""")

┌───────────┬─────────────────────┬──────────────────────────┬────────────────────────────┬─────────────────────────────┐
│ row_count │ rows_after_midnight │ rows_before_service_date │ rows_more_than_1_day_later │ origins_not_on_service_date │
│   int64   │        int64        │          int64           │           int64            │            int64            │
├───────────┼─────────────────────┼──────────────────────────┼────────────────────────────┼─────────────────────────────┤
│      2460 │                 516 │                        0 │                          0 │                           0 │
└───────────┴─────────────────────┴──────────────────────────┴────────────────────────────┴─────────────────────────────┘



### 6.3 Order of timestamps

Departure cannot precede arrival at a stop. Arrival cannot precede the previous departure.

In [16]:
show("""
    -- LAG returns NULL at the origin, so origin rows can never be flagged by the between-stop rules.
    WITH ordered AS (
        SELECT
            *,
            LAG(scheduled_departure) OVER run AS previous_scheduled_departure,
            LAG(actual_departure) OVER run AS previous_actual_departure
        FROM events_checked
        WINDOW run AS (PARTITION BY service_id ORDER BY station_sequence)
    )
    SELECT 'Scheduled departure before scheduled arrival' AS check_name, 'within a stop' AS scope,
        COUNT(*) FILTER (WHERE scheduled_departure < scheduled_arrival) AS rows_flagged
    FROM ordered
    UNION ALL
    SELECT 'Actual departure before actual arrival', 'within a stop',
        COUNT(*) FILTER (WHERE actual_departure < actual_arrival)
    FROM ordered
    UNION ALL
    SELECT 'Scheduled arrival before previous scheduled departure', 'between stops',
        COUNT(*) FILTER (WHERE scheduled_arrival < previous_scheduled_departure)
    FROM ordered
    UNION ALL
    SELECT 'Actual arrival before previous actual departure', 'between stops',
        COUNT(*) FILTER (WHERE actual_arrival < previous_actual_departure)
    FROM ordered
""")

┌───────────────────────────────────────────────────────┬───────────────┬──────────────┐
│                      check_name                       │     scope     │ rows_flagged │
│                        varchar                        │    varchar    │    int64     │
├───────────────────────────────────────────────────────┼───────────────┼──────────────┤
│ Scheduled departure before scheduled arrival          │ within a stop │            0 │
│ Actual departure before actual arrival                │ within a stop │           42 │
│ Scheduled arrival before previous scheduled departure │ between stops │            0 │
│ Actual arrival before previous actual departure       │ between stops │            0 │
└───────────────────────────────────────────────────────┴───────────────┴──────────────┘



### 6.4 Negative dwell

Split by position: at the origin it is an artefact of the placeholder arrival, at intermediate stops it is a genuine conflict.

In [17]:
show("""
    SELECT
        -- Numeric prefix only fixes the display order.
        CASE WHEN is_origin THEN '1 origin' WHEN is_terminus THEN '3 terminus' ELSE '2 intermediate' END AS stop_position,
        COUNT(*) AS stops,
        COUNT(*) FILTER (WHERE actual_dwell_min < 0) AS negative_dwell_rows,
        COUNT(*) FILTER (WHERE actual_dwell_min = 0) AS zero_dwell_rows,
        MIN(actual_dwell_min) AS min_dwell_min,
        MAX(actual_dwell_min) AS max_dwell_min
    FROM events_checked
    GROUP BY stop_position
    ORDER BY stop_position
""")

┌────────────────┬───────┬─────────────────────┬─────────────────┬───────────────┬───────────────┐
│ stop_position  │ stops │ negative_dwell_rows │ zero_dwell_rows │ min_dwell_min │ max_dwell_min │
│    varchar     │ int64 │        int64        │      int64      │     int64     │     int64     │
├────────────────┼───────┼─────────────────────┼─────────────────┼───────────────┼───────────────┤
│ 1 origin       │   426 │                  29 │              29 │            -3 │            14 │
│ 2 intermediate │  1608 │                  13 │              29 │            -3 │            16 │
│ 3 terminus     │   426 │                   0 │               0 │             2 │             7 │
└────────────────┴───────┴─────────────────────┴─────────────────┴───────────────┴───────────────┘



In [18]:
# Row-level detail for the genuine cases, so each can be traced back to the source file.
show("""
    SELECT
        service_id, train_id, station_sequence, station_id,
        actual_arrival, actual_departure, actual_dwell_min,
        arrival_delay_min, departure_delay_min
    FROM events_checked
    WHERE actual_dwell_min < 0 AND NOT is_origin AND NOT is_terminus
    ORDER BY actual_dwell_min, service_id
""")

┌────────────┬──────────┬──────────────────┬────────────┬─────────────────────┬─────────────────────┬──────────────────┬───────────────────┬─────────────────────┐
│ service_id │ train_id │ station_sequence │ station_id │   actual_arrival    │  actual_departure   │ actual_dwell_min │ arrival_delay_min │ departure_delay_min │
│  varchar   │ varchar  │      int64       │  varchar   │      timestamp      │      timestamp      │      int64       │       int64       │        int64        │
├────────────┼──────────┼──────────────────┼────────────┼─────────────────────┼─────────────────────┼──────────────────┼───────────────────┼─────────────────────┤
│ RUN000191  │ TR005    │                2 │ ST002      │ 2026-08-20 07:08:00 │ 2026-08-20 07:05:00 │               -3 │                 8 │                   3 │
│ RUN000196  │ TR005    │                5 │ ST019      │ 2026-08-26 11:08:00 │ 2026-08-26 11:06:00 │               -2 │                 7 │                   3 │
│ RUN000131  │ TR004  

### 6.5 Origin and terminus placeholders

Origin arrival and terminus departure cannot be observations. This shows what the file stores instead.

In [19]:
show("""
    -- Profile how each placeholder is populated before deciding how to treat it downstream.
    SELECT
        'origin' AS stop_position,
        'arrival' AS placeholder_field,
        COUNT(*) AS stops,
        COUNT(*) FILTER (WHERE scheduled_arrival = scheduled_departure) AS scheduled_arrival_equals_departure,
        COUNT(*) FILTER (WHERE actual_arrival = scheduled_arrival) AS actual_equals_scheduled,
        COUNT(*) FILTER (WHERE arrival_delay_min <> 0) AS rows_with_nonzero_delay
    FROM events_checked
    WHERE is_origin
    UNION ALL
    SELECT 'terminus', 'departure', COUNT(*),
        COUNT(*) FILTER (WHERE scheduled_arrival = scheduled_departure),
        COUNT(*) FILTER (WHERE actual_departure = scheduled_departure),
        COUNT(*) FILTER (WHERE departure_delay_min <> 0)
    FROM events_checked
    WHERE is_terminus
""")

┌───────────────┬───────────────────┬───────┬────────────────────────────────────┬─────────────────────────┬─────────────────────────┐
│ stop_position │ placeholder_field │ stops │ scheduled_arrival_equals_departure │ actual_equals_scheduled │ rows_with_nonzero_delay │
│    varchar    │      varchar      │ int64 │               int64                │          int64          │          int64          │
├───────────────┼───────────────────┼───────┼────────────────────────────────────┼─────────────────────────┼─────────────────────────┤
│ origin        │ arrival           │   426 │                                426 │                     426 │                       0 │
│ terminus      │ departure         │   426 │                                  0 │                      28 │                     398 │
└───────────────┴───────────────────┴───────┴────────────────────────────────────┴─────────────────────────┴─────────────────────────┘



Origin arrival is a clean placeholder. Terminus departure is not: it is stored a few minutes after arrival, which contradicts `data/README.md`.

In [20]:
show("""
    -- Sample only: the full count is in the previous query.
    SELECT
        service_id, station_sequence, station_id,
        scheduled_arrival, scheduled_departure, scheduled_dwell_min,
        arrival_delay_min, departure_delay_min
    FROM events_checked
    WHERE is_terminus
    ORDER BY service_id
    LIMIT 5
""")

┌────────────┬──────────────────┬────────────┬─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬─────────────────────┐
│ service_id │ station_sequence │ station_id │  scheduled_arrival  │ scheduled_departure │ scheduled_dwell_min │ arrival_delay_min │ departure_delay_min │
│  varchar   │      int64       │  varchar   │      timestamp      │      timestamp      │        int64        │       int64       │        int64        │
├────────────┼──────────────────┼────────────┼─────────────────────┼─────────────────────┼─────────────────────┼───────────────────┼─────────────────────┤
│ RUN000001  │                5 │ ST011      │ 2026-08-01 11:19:00 │ 2026-08-01 11:25:00 │                   6 │                24 │                  24 │
│ RUN000002  │                5 │ ST011      │ 2026-08-02 11:17:00 │ 2026-08-02 11:23:00 │                   6 │                 8 │                   8 │
│ RUN000003  │                5 │ ST011      │ 2026-08-03 11:15:00 │ 2

## 7. Delays

### 7.1 Stored delays against the timestamps

Stored delays are derived fields and must equal actual minus scheduled time.

In [21]:
show("""
    -- Exact comparison, no tolerance: both sides are whole minutes.
    SELECT
        'arrival_delay_min' AS delay_column,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE arrival_delay_min <> calc_arrival_delay_min) AS rows_not_matching_timestamps
    FROM events_checked
    UNION ALL
    SELECT 'departure_delay_min', COUNT(*),
        COUNT(*) FILTER (WHERE departure_delay_min <> calc_departure_delay_min)
    FROM events_checked
""")

┌─────────────────────┬───────────┬──────────────────────────────┐
│    delay_column     │ row_count │ rows_not_matching_timestamps │
│       varchar       │   int64   │            int64             │
├─────────────────────┼───────────┼──────────────────────────────┤
│ arrival_delay_min   │      2460 │                            0 │
│ departure_delay_min │      2460 │                            0 │
└─────────────────────┴───────────┴──────────────────────────────┘



### 7.2 Delay distribution

Origin arrivals and terminus departures are excluded. Outlier threshold: Tukey upper fence, Q3 + 1.5 × IQR.

In [22]:
con.execute("""
    -- Long format, one row per stop per measure. Placeholder rows are excluded here once, so no later query has to filter them.
    CREATE OR REPLACE TEMP VIEW delay_measures AS
    SELECT service_id, train_id, station_sequence, station_id, simulated_disruption_flag,
        simulated_disruption_type, 'arrival' AS measure, arrival_delay_min AS delay_min
    FROM events_checked
    WHERE NOT is_origin
    UNION ALL
    SELECT service_id, train_id, station_sequence, station_id, simulated_disruption_flag,
        simulated_disruption_type, 'departure', departure_delay_min
    FROM events_checked
    WHERE NOT is_terminus
""")

con.execute("""
    -- Tukey upper fence (Q3 + 1.5 * IQR): derived from the data, so it needs no retuning when the data changes.
    CREATE OR REPLACE TEMP VIEW delay_fences AS
    SELECT
        measure,
        quantile_cont(delay_min, 0.25) AS q1,
        quantile_cont(delay_min, 0.75) AS q3,
        quantile_cont(delay_min, 0.75)
            + 1.5 * (quantile_cont(delay_min, 0.75) - quantile_cont(delay_min, 0.25)) AS upper_fence_min
    FROM delay_measures
    GROUP BY measure
""")

show("""
    SELECT
        m.measure,
        COUNT(*) AS stops_measured,
        MIN(delay_min) AS min_delay,
        ANY_VALUE(f.q1) AS q1,
        MEDIAN(delay_min) AS median_delay,
        ANY_VALUE(f.q3) AS q3,
        quantile_cont(delay_min, 0.99) AS p99,
        MAX(delay_min) AS max_delay,
        ROUND(AVG(delay_min), 2) AS mean_delay,
        ANY_VALUE(f.upper_fence_min) AS upper_fence_min
    FROM delay_measures m
    JOIN delay_fences f USING (measure)
    GROUP BY m.measure
    ORDER BY m.measure
""")

┌───────────┬────────────────┬───────────┬────────┬──────────────┬────────┬────────┬───────────┬────────────┬─────────────────┐
│  measure  │ stops_measured │ min_delay │   q1   │ median_delay │   q3   │  p99   │ max_delay │ mean_delay │ upper_fence_min │
│  varchar  │     int64      │   int64   │ double │    double    │ double │ double │   int64   │   double   │     double      │
├───────────┼────────────────┼───────────┼────────┼──────────────┼────────┼────────┼───────────┼────────────┼─────────────────┤
│ arrival   │           2034 │        -2 │   -1.0 │          2.0 │    5.0 │   21.0 │        33 │        2.7 │            14.0 │
│ departure │           2034 │        -3 │    1.0 │          4.0 │    6.0 │   20.0 │        36 │       4.05 │            13.5 │
└───────────┴────────────────┴───────────┴────────┴──────────────┴────────┴────────┴───────────┴────────────┴─────────────────┘



### 7.3 Negative delays

Early running is valid. Magnitude matters more than count.

In [23]:
show("""
    SELECT
        measure,
        delay_min,
        COUNT(*) AS stops,
        ROUND(100.0 * COUNT(*) / ANY_VALUE(stops_measured), 1) AS pct_of_measured_stops
    -- stops_measured is taken before the filter, so the percentage is of all measured stops.
    FROM (
        SELECT measure, delay_min, COUNT(*) OVER (PARTITION BY measure) AS stops_measured
        FROM delay_measures
    )
    WHERE delay_min < 0
    GROUP BY measure, delay_min
    ORDER BY measure, delay_min
""")

┌───────────┬───────────┬───────┬───────────────────────┐
│  measure  │ delay_min │ stops │ pct_of_measured_stops │
│  varchar  │   int64   │ int64 │        double         │
├───────────┼───────────┼───────┼───────────────────────┤
│ arrival   │        -2 │   413 │                  20.3 │
│ arrival   │        -1 │   173 │                   8.5 │
│ departure │        -3 │   119 │                   5.9 │
│ departure │        -2 │    82 │                   4.0 │
│ departure │        -1 │   107 │                   5.3 │
└───────────┴───────────┴───────┴───────────────────────┘



In [24]:
# Early-running rate per measure. The denominator already excludes placeholder rows.
show("""
    SELECT
        measure,
        COUNT(*) AS stops_measured,
        COUNT(*) FILTER (WHERE delay_min < 0) AS early_stops,
        ROUND(100.0 * COUNT(*) FILTER (WHERE delay_min < 0) / COUNT(*), 1) AS early_pct,
        MIN(delay_min) AS earliest_min
    FROM delay_measures
    GROUP BY measure
    ORDER BY measure
""")

┌───────────┬────────────────┬─────────────┬───────────┬──────────────┐
│  measure  │ stops_measured │ early_stops │ early_pct │ earliest_min │
│  varchar  │     int64      │    int64    │  double   │    int64     │
├───────────┼────────────────┼─────────────┼───────────┼──────────────┤
│ arrival   │           2034 │         586 │      28.8 │           -2 │
│ departure │           2034 │         308 │      15.1 │           -3 │
└───────────┴────────────────┴─────────────┴───────────┴──────────────┘



### 7.4 Unusually large delays

Delays above the upper fence, with a fixed 30 minute band for reference.

In [25]:
show("""
    SELECT
        m.measure,
        ANY_VALUE(f.upper_fence_min) AS upper_fence_min,
        COUNT(*) FILTER (WHERE m.delay_min > f.upper_fence_min) AS stops_above_fence,
        ROUND(100.0 * COUNT(*) FILTER (WHERE m.delay_min > f.upper_fence_min) / COUNT(*), 1) AS pct_above_fence,
        -- Fixed 30 minute band as a stable reference next to the statistical fence.
        COUNT(*) FILTER (WHERE m.delay_min >= 30) AS stops_30_min_or_more,
        MAX(m.delay_min) AS max_delay
    FROM delay_measures m
    JOIN delay_fences f USING (measure)
    GROUP BY m.measure
    ORDER BY m.measure
""")

┌───────────┬─────────────────┬───────────────────┬─────────────────┬──────────────────────┬───────────┐
│  measure  │ upper_fence_min │ stops_above_fence │ pct_above_fence │ stops_30_min_or_more │ max_delay │
│  varchar  │     double      │       int64       │     double      │        int64         │   int64   │
├───────────┼─────────────────┼───────────────────┼─────────────────┼──────────────────────┼───────────┤
│ arrival   │            14.0 │                51 │             2.5 │                    7 │        33 │
│ departure │            13.5 │                74 │             3.6 │                    2 │        36 │
└───────────┴─────────────────┴───────────────────┴─────────────────┴──────────────────────┴───────────┘



In [26]:
# Largest delays with run and stop, for a plausibility review against the disruption fields.
show("""
    SELECT
        m.service_id, m.train_id, m.station_sequence, m.station_id, m.measure, m.delay_min,
        m.simulated_disruption_flag AS disruption, m.simulated_disruption_type AS disruption_type
    FROM delay_measures m
    ORDER BY m.delay_min DESC, m.service_id, m.station_sequence
    LIMIT 15
""")

┌────────────┬──────────┬──────────────────┬────────────┬───────────┬───────────┬────────────┬─────────────────┐
│ service_id │ train_id │ station_sequence │ station_id │  measure  │ delay_min │ disruption │ disruption_type │
│  varchar   │ varchar  │      int64       │  varchar   │  varchar  │   int64   │  boolean   │     varchar     │
├────────────┼──────────┼──────────────────┼────────────┼───────────┼───────────┼────────────┼─────────────────┤
│ RUN000367  │ TR009    │                4 │ ST014      │ departure │        36 │ true       │ track           │
│ RUN000367  │ TR009    │                5 │ ST015      │ arrival   │        33 │ true       │ track           │
│ RUN000367  │ TR009    │                6 │ ST001      │ arrival   │        32 │ true       │ track           │
│ RUN000073  │ TR002    │                6 │ ST011      │ arrival   │        31 │ true       │ operational     │
│ RUN000367  │ TR009    │                5 │ ST015      │ departure │        31 │ true       │ t

### 7.5 Large delays and the disruption flag

An unflagged outlier is a prompt to review, not proof of an error.

In [27]:
show("""
    -- An unflagged outlier is a prompt to review, not proof of an error.
    SELECT
        m.measure,
        m.simulated_disruption_flag AS disruption_flag,
        COUNT(*) AS stops_measured,
        ROUND(AVG(m.delay_min), 2) AS mean_delay,
        MAX(m.delay_min) AS max_delay,
        COUNT(*) FILTER (WHERE m.delay_min > f.upper_fence_min) AS stops_above_fence
    FROM delay_measures m
    JOIN delay_fences f USING (measure)
    GROUP BY m.measure, m.simulated_disruption_flag
    ORDER BY m.measure, m.simulated_disruption_flag
""")

┌───────────┬─────────────────┬────────────────┬────────────┬───────────┬───────────────────┐
│  measure  │ disruption_flag │ stops_measured │ mean_delay │ max_delay │ stops_above_fence │
│  varchar  │     boolean     │     int64      │   double   │   int64   │       int64       │
├───────────┼─────────────────┼────────────────┼────────────┼───────────┼───────────────────┤
│ arrival   │ false           │           1898 │       1.95 │        15 │                 1 │
│ arrival   │ true            │            136 │      13.11 │        33 │                50 │
│ departure │ false           │           1926 │       3.51 │        21 │                26 │
│ departure │ true            │            108 │      13.81 │        36 │                48 │
└───────────┴─────────────────┴────────────────┴────────────┴───────────┴───────────────────┘



## 8. Consistency between the tables

### 8.1 Events without a master record

Every `train_id` and `station_id` in the events must resolve to a master record.

In [28]:
show("""
    -- LEFT JOIN keeps every event row: an unmatched foreign key surfaces as NULL on the master side.
    SELECT
        COUNT(*) AS event_rows,
        COUNT(*) FILTER (WHERE t.train_id IS NULL) AS rows_with_unknown_train_id,
        COUNT(*) FILTER (WHERE s.station_id IS NULL) AS rows_with_unknown_station_id,
        COUNT(DISTINCT e.train_id) FILTER (WHERE t.train_id IS NULL) AS unknown_train_ids,
        COUNT(DISTINCT e.station_id) FILTER (WHERE s.station_id IS NULL) AS unknown_station_ids
    FROM train_run_events e
    LEFT JOIN train_master t ON t.train_id = e.train_id
    LEFT JOIN station_master s ON s.station_id = e.station_id
""")

┌────────────┬────────────────────────────┬──────────────────────────────┬───────────────────┬─────────────────────┐
│ event_rows │ rows_with_unknown_train_id │ rows_with_unknown_station_id │ unknown_train_ids │ unknown_station_ids │
│   int64    │           int64            │            int64             │       int64       │        int64        │
├────────────┼────────────────────────────┼──────────────────────────────┼───────────────────┼─────────────────────┤
│       2460 │                          0 │                            0 │                 0 │                   0 │
└────────────┴────────────────────────────┴──────────────────────────────┴───────────────────┴─────────────────────┘



### 8.2 Names that disagree with the master tables

The events table repeats names held in the master tables. They must agree.

In [29]:
show("""
    -- IS DISTINCT FROM is NULL-safe: a NULL on one side counts as a mismatch instead of dropping out.
    SELECT
        'train_name' AS column_name,
        COUNT(*) AS rows_compared,
        COUNT(*) FILTER (WHERE e.train_name IS DISTINCT FROM t.train_name) AS rows_with_different_name,
        COUNT(DISTINCT e.train_name) AS names_in_events,
        COUNT(DISTINCT t.train_name) AS names_in_master
    FROM train_run_events e
    JOIN train_master t ON t.train_id = e.train_id
    UNION ALL
    SELECT 'station_name', COUNT(*),
        COUNT(*) FILTER (WHERE e.station_name IS DISTINCT FROM s.station_name),
        COUNT(DISTINCT e.station_name),
        COUNT(DISTINCT s.station_name)
    FROM train_run_events e
    JOIN station_master s ON s.station_id = e.station_id
""")

┌──────────────┬───────────────┬──────────────────────────┬─────────────────┬─────────────────┐
│ column_name  │ rows_compared │ rows_with_different_name │ names_in_events │ names_in_master │
│   varchar    │     int64     │          int64           │      int64      │      int64      │
├──────────────┼───────────────┼──────────────────────────┼─────────────────┼─────────────────┤
│ train_name   │          2460 │                        0 │              10 │              10 │
│ station_name │          2460 │                        0 │              19 │              19 │
└──────────────┴───────────────┴──────────────────────────┴─────────────────┴─────────────────┘



### 8.3 Master records that are never used

Not a defect, but it appears as an empty row in master-driven reports.

In [30]:
show("""
    -- COUNT(e.<col>) counts matched rows only, so an unused master record yields 0, not 1.
    SELECT 'train_master' AS master_table, t.train_id AS id, t.train_name AS name, COUNT(e.train_id) AS event_rows
    FROM train_master t
    LEFT JOIN train_run_events e ON e.train_id = t.train_id
    GROUP BY t.train_id, t.train_name
    HAVING COUNT(e.train_id) = 0
    UNION ALL
    SELECT 'station_master', s.station_id, s.station_name, COUNT(e.station_id)
    FROM station_master s
    LEFT JOIN train_run_events e ON e.station_id = s.station_id
    GROUP BY s.station_id, s.station_name
    HAVING COUNT(e.station_id) = 0
""")

┌────────────────┬─────────┬─────────────────────────┬────────────┐
│  master_table  │   id    │          name           │ event_rows │
│    varchar     │ varchar │         varchar         │   int64    │
├────────────────┼─────────┼─────────────────────────┼────────────┤
│ station_master │ ST006   │ Bangabandhu Bridge East │          0 │
└────────────────┴─────────┴─────────────────────────┴────────────┘



### 8.4 Route endpoints against `train_master`

Endpoint names must resolve in `station_master` and match the first and last stop of every run.

In [31]:
show("""
    WITH run_endpoints AS (
        SELECT
            service_id,
            train_id,
            arg_min(station_name, station_sequence) AS first_stop,
            arg_max(station_name, station_sequence) AS last_stop
        FROM train_run_events
        GROUP BY service_id, train_id
    )
    SELECT
        t.train_id,
        t.origin_station,
        o.station_id AS origin_station_id,
        t.destination_station,
        d.station_id AS destination_station_id,
        COUNT(r.service_id) AS runs,
        COUNT(*) FILTER (WHERE r.first_stop IS DISTINCT FROM t.origin_station) AS runs_with_other_origin,
        COUNT(*) FILTER (WHERE r.last_stop IS DISTINCT FROM t.destination_station) AS runs_with_other_destination
    FROM train_master t
    -- train_master stores endpoint names, not IDs. A NULL station ID means the name does not resolve.
    LEFT JOIN station_master o ON o.station_name = t.origin_station
    LEFT JOIN station_master d ON d.station_name = t.destination_station
    LEFT JOIN run_endpoints r ON r.train_id = t.train_id
    GROUP BY ALL
    ORDER BY t.train_id
""")

┌──────────┬────────────────┬───────────────────┬─────────────────────┬────────────────────────┬───────┬────────────────────────┬─────────────────────────────┐
│ train_id │ origin_station │ origin_station_id │ destination_station │ destination_station_id │ runs  │ runs_with_other_origin │ runs_with_other_destination │
│ varchar  │    varchar     │      varchar      │       varchar       │        varchar         │ int64 │         int64          │            int64            │
├──────────┼────────────────┼───────────────────┼─────────────────────┼────────────────────────┼───────┼────────────────────────┼─────────────────────────────┤
│ TR001    │ Dhaka          │ ST001             │ Chattogram          │ ST011                  │    39 │                      0 │                           0 │
│ TR002    │ Dhaka          │ ST001             │ Chattogram          │ ST011                  │    45 │                      0 │                           0 │
│ TR003    │ Dhaka          │ ST001     

## 9. Service calendar coverage

Expected train × date grid against actual runs. A fixed weekday gap is a day off, scattered gaps suggest lost data.

In [32]:
show("""
    -- Build the expected train x date grid first: absent rows are only detectable against an expected set.
    WITH calendar AS (
        SELECT UNNEST(generate_series(
            (SELECT MIN(service_date) FROM train_run_events),
            (SELECT MAX(service_date) FROM train_run_events),
            INTERVAL 1 DAY
        ))::DATE AS service_date
    ),
    runs AS (
        SELECT DISTINCT train_id, service_date FROM train_run_events
    )
    SELECT
        t.train_id,
        COUNT(*) AS days_in_period,
        COUNT(r.service_date) AS days_with_a_run,
        COUNT(*) - COUNT(r.service_date) AS days_without_a_run,
        COALESCE(string_agg(DISTINCT dayname(c.service_date), ', ') FILTER (WHERE r.service_date IS NULL), '-') AS weekdays_without_a_run
    FROM train_master t
    CROSS JOIN calendar c
    LEFT JOIN runs r ON r.train_id = t.train_id AND r.service_date = c.service_date
    GROUP BY t.train_id
    ORDER BY t.train_id
""")

┌──────────┬────────────────┬─────────────────┬────────────────────┬────────────────────────┐
│ train_id │ days_in_period │ days_with_a_run │ days_without_a_run │ weekdays_without_a_run │
│ varchar  │     int64      │      int64      │       int64        │        varchar         │
├──────────┼────────────────┼─────────────────┼────────────────────┼────────────────────────┤
│ TR001    │             45 │              39 │                  6 │ Friday                 │
│ TR002    │             45 │              45 │                  0 │ -                      │
│ TR003    │             45 │              45 │                  0 │ -                      │
│ TR004    │             45 │              45 │                  0 │ -                      │
│ TR005    │             45 │              39 │                  6 │ Tuesday                │
│ TR006    │             45 │              39 │                  6 │ Friday                 │
│ TR007    │             45 │              39 │             

## 10. Summary

### 10.1 Scorecard

Recomputed on every run. **PASS** nothing flagged, **REVIEW** decision needed, **INFO** expected behaviour.

In [33]:
show("""
    -- One scalar subquery per check, recomputed from the tables on every run. Nothing here is hard-coded.
    WITH checks AS (
        -- kind separates defects ('error') from expected behaviour that is only counted ('info').
        SELECT 1 AS check_no, 'Missing values' AS category, 'NULL or blank values, all tables' AS check_name, 'error' AS kind, (
            (SELECT COUNT(*) FROM (SELECT COLUMNS(*)::VARCHAR FROM train_run_events)
                UNPIVOT INCLUDE NULLS (value FOR column_name IN (COLUMNS(*))) WHERE value IS NULL OR trim(value) = '')
          + (SELECT COUNT(*) FROM (SELECT COLUMNS(*)::VARCHAR FROM train_master)
                UNPIVOT INCLUDE NULLS (value FOR column_name IN (COLUMNS(*))) WHERE value IS NULL OR trim(value) = '')
          + (SELECT COUNT(*) FROM (SELECT COLUMNS(*)::VARCHAR FROM station_master)
                UNPIVOT INCLUDE NULLS (value FOR column_name IN (COLUMNS(*))) WHERE value IS NULL OR trim(value) = '')
        ) AS rows_flagged
        UNION ALL
        SELECT 2, 'Missing values', 'Disruption flag and type disagree', 'error',
            (SELECT COUNT(*) FROM train_run_events
             WHERE simulated_disruption_flag <> (simulated_disruption_type <> 'none'))
        UNION ALL
        SELECT 3, 'Duplicates', 'Duplicate keys, all tables', 'error', (
            (SELECT COUNT(*) - COUNT(DISTINCT (service_id, station_sequence)) FROM train_run_events)
          + (SELECT COUNT(*) - COUNT(DISTINCT train_id) FROM train_master)
          + (SELECT COUNT(*) - COUNT(DISTINCT station_id) FROM station_master)
        )
        UNION ALL
        SELECT 4, 'Duplicates', 'Station repeated within a run', 'error',
            (SELECT COUNT(*) FROM (SELECT 1 FROM train_run_events GROUP BY service_id, station_id HAVING COUNT(*) > 1))
        UNION ALL
        SELECT 5, 'Identifiers', 'IDs not in the expected format', 'error', (
            (SELECT COUNT(*) FROM train_run_events
             WHERE NOT regexp_full_match(service_id, 'RUN[0-9]{6}')
                OR NOT regexp_full_match(train_id, 'TR[0-9]{3}')
                OR NOT regexp_full_match(station_id, 'ST[0-9]{3}'))
          + (SELECT COUNT(*) FROM train_master WHERE NOT regexp_full_match(train_id, 'TR[0-9]{3}'))
          + (SELECT COUNT(*) FROM station_master WHERE NOT regexp_full_match(station_id, 'ST[0-9]{3}'))
        )
        UNION ALL
        SELECT 6, 'Identifiers', 'Runs with mixed trains, mixed dates or sequence gaps', 'error',
            (SELECT COUNT(*) FROM (
                SELECT 1 FROM train_run_events GROUP BY service_id
                HAVING COUNT(DISTINCT train_id) > 1 OR COUNT(DISTINCT service_date) > 1
                    OR MIN(station_sequence) <> 1 OR MAX(station_sequence) <> COUNT(*)))
        UNION ALL
        SELECT 7, 'Timestamps', 'Timestamp outside service date or the day after', 'error',
            (SELECT COUNT(*) FROM events_checked
             WHERE scheduled_arrival::DATE NOT IN (service_date, service_date + 1))
        UNION ALL
        SELECT 8, 'Timestamps', 'Arrival before departure from the previous stop', 'error',
            (SELECT COUNT(*) FROM (
                SELECT actual_arrival, scheduled_arrival,
                    LAG(actual_departure) OVER run AS previous_actual_departure,
                    LAG(scheduled_departure) OVER run AS previous_scheduled_departure
                FROM train_run_events
                WINDOW run AS (PARTITION BY service_id ORDER BY station_sequence))
             WHERE actual_arrival < previous_actual_departure
                OR scheduled_arrival < previous_scheduled_departure)
        UNION ALL
        SELECT 9, 'Timestamps', 'Negative dwell at intermediate stops', 'error',
            (SELECT COUNT(*) FROM events_checked
             WHERE actual_dwell_min < 0 AND NOT is_origin AND NOT is_terminus)
        UNION ALL
        SELECT 10, 'Timestamps', 'Negative dwell at origin stops (placeholder arrival)', 'error',
            (SELECT COUNT(*) FROM events_checked WHERE actual_dwell_min < 0 AND is_origin)
        UNION ALL
        SELECT 11, 'Timestamps', 'Terminus rows with a departure after arrival', 'error',
            (SELECT COUNT(*) FROM events_checked WHERE is_terminus AND actual_departure <> actual_arrival)
        UNION ALL
        SELECT 12, 'Delays', 'Stored delay differs from the timestamps', 'error',
            (SELECT COUNT(*) FROM events_checked
             WHERE arrival_delay_min <> calc_arrival_delay_min
                OR departure_delay_min <> calc_departure_delay_min)
        UNION ALL
        SELECT 13, 'Delays', 'Negative delays (early running)', 'info',
            (SELECT COUNT(*) FROM delay_measures WHERE delay_min < 0)
        UNION ALL
        SELECT 14, 'Delays', 'Delays above the upper fence', 'info',
            (SELECT COUNT(*) FROM delay_measures m JOIN delay_fences f USING (measure)
             WHERE m.delay_min > f.upper_fence_min)
        UNION ALL
        SELECT 15, 'Table consistency', 'Event rows with an unknown train or station ID', 'error',
            (SELECT COUNT(*) FROM train_run_events e
             LEFT JOIN train_master t ON t.train_id = e.train_id
             LEFT JOIN station_master s ON s.station_id = e.station_id
             WHERE t.train_id IS NULL OR s.station_id IS NULL)
        UNION ALL
        SELECT 16, 'Table consistency', 'Event rows whose name differs from the master', 'error',
            (SELECT COUNT(*) FROM train_run_events e
             JOIN train_master t ON t.train_id = e.train_id
             JOIN station_master s ON s.station_id = e.station_id
             WHERE e.train_name IS DISTINCT FROM t.train_name
                OR e.station_name IS DISTINCT FROM s.station_name)
        UNION ALL
        SELECT 17, 'Table consistency', 'Master records with no events', 'info', (
            (SELECT COUNT(*) FROM train_master t
             LEFT JOIN train_run_events e ON e.train_id = t.train_id WHERE e.train_id IS NULL)
          + (SELECT COUNT(*) FROM station_master s
             LEFT JOIN train_run_events e ON e.station_id = s.station_id WHERE e.station_id IS NULL)
        )
    )
    SELECT
        check_no,
        category,
        check_name,
        rows_flagged,
        CASE
            WHEN rows_flagged = 0 THEN 'PASS'
            WHEN kind = 'info' THEN 'INFO'
            ELSE 'REVIEW'
        END AS status
    FROM checks
    ORDER BY check_no
""")

┌──────────┬───────────────────┬──────────────────────────────────────────────────────┬──────────────┬─────────┐
│ check_no │     category      │                      check_name                      │ rows_flagged │ status  │
│  int32   │      varchar      │                       varchar                        │    int64     │ varchar │
├──────────┼───────────────────┼──────────────────────────────────────────────────────┼──────────────┼─────────┤
│        1 │ Missing values    │ NULL or blank values, all tables                     │            0 │ PASS    │
│        2 │ Missing values    │ Disruption flag and type disagree                    │            0 │ PASS    │
│        3 │ Duplicates        │ Duplicate keys, all tables                           │            0 │ PASS    │
│        4 │ Duplicates        │ Station repeated within a run                        │            0 │ PASS    │
│        5 │ Identifiers       │ IDs not in the expected format                       │         

In [34]:
# Temporary views are session-scoped and are dropped with the connection.
con.close()
print("Connection closed.")

Connection closed.
